<div style='background:#0f172a;padding:28px 32px;border-radius:12px;margin-bottom:8px'>
<h1 style='color:white;margin:0 0 8px 0;font-size:22px'>EDA — Análisis de intensidad para segmentación clásica (BraTS 2024 GLI)</h1>
<h2 style='color:#94a3b8;margin:0 0 12px 0;font-size:15px;font-weight:400'>Secciones A–F · Complemento orientado a métodos clásicos con ITK (Otsu, K-means/GMM, crecimiento de regiones, watershed)</h2>
<div style='color:#64748b;font-size:13px'>Abel Albuez · Victoria Acero · Santiago Gil</div>
</div>

Este notebook complementa el EDA de la entrega 2 con el análisis de **dominio de intensidad**, que es lo que determina si los métodos clásicos van a funcionar. **No usa SimpleITK** (solo numpy/scipy/scikit-image/nibabel/plotly), así que corre directo en Colab sin el error de módulo.

**Cómo usarlo:** se puede correr de dos formas:
1. **Standalone (recomendado):** ejecuta *Run all*. Monta Drive, carga `df_vol` desde tu checkpoint `EDA_volumenes_completo.csv`, y **extrae del ZIP solo los ~25 casos de la muestra** (no los 2,200), así que es rápido.
2. **Anexado a tu EDA original:** pega estas celdas después de tus secciones S1–S11; reutiliza el `df_vol` que ya tengas en memoria.

Ajusta `DRIVE_DIR`, `DATASET_DIR` y `VOL_CSV` en la celda *Fuente de datos* si tus rutas difieren.

In [ ]:
# === Setup (sin SimpleITK) ===
# scikit-image, scipy, plotly y numpy ya vienen en Colab; solo aseguramos nibabel.
import importlib, subprocess, sys
for pkg, mod in [('nibabel','nibabel')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'])

import os, glob, gc, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as spstats
from scipy import ndimage
from skimage.filters import threshold_multiotsu

print('Librerías cargadas (sin SimpleITK)')

In [ ]:
# === Fuente de datos (Drive) ===
import zipfile, shutil

# Montaje robusto: si el mount está caído, re-montar.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except Exception:
    EN_COLAB = False

# >>> AJUSTA ESTAS RUTAS SI HACE FALTA <<<
DRIVE_DIR   = '/content/drive/MyDrive/BRATS-2024'                    # carpeta con el ZIP y los checkpoints
DATASET_DIR = '/content/brats2024'                                  # dónde se extraen/buscan los NIfTI
VOL_CSV     = os.path.join(DRIVE_DIR, 'EDA_volumenes_completo.csv')  # checkpoint de volúmenes (entrega 2)
COPIAR_ZIP_LOCAL = True   # copia el ZIP a disco local (evita 'Transport endpoint is not connected')

if EN_COLAB:
    try:
        os.listdir('/content/drive/MyDrive')          # prueba de conectividad del mount
    except Exception:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)

def _encontrar_zip(d):
    zips = sorted(glob.glob(os.path.join(d, '*.zip')))
    if not zips: return None
    return next((z for z in zips if 'TrainingData' in os.path.basename(z)), max(zips, key=os.path.getsize))

TRAINING_ZIP = _encontrar_zip(DRIVE_DIR)

# Copiar el ZIP de Drive a disco local UNA vez (lectura secuencial estable).
if TRAINING_ZIP and COPIAR_ZIP_LOCAL and TRAINING_ZIP.startswith('/content/drive'):
    _local = os.path.join('/content', os.path.basename(TRAINING_ZIP))
    try:
        _ok = os.path.exists(_local) and os.path.getsize(_local) == os.path.getsize(TRAINING_ZIP)
    except Exception:
        _ok = os.path.exists(_local)
    if not _ok:
        print('Copiando ZIP de Drive a disco local (una sola vez)... puede tardar unos minutos.')
        shutil.copy(TRAINING_ZIP, _local)
    TRAINING_ZIP = _local
    print('ZIP local en uso:', TRAINING_ZIP)

print('Drive montado:', EN_COLAB)
print('ZIP del dataset:', TRAINING_ZIP)
print('Checkpoint de volúmenes existe:', os.path.exists(VOL_CSV))

In [ ]:
# === Constantes (se definen solo si no existen, para no pisar tu EDA original) ===
LABEL_MAP        = globals().get('LABEL_MAP', {1:'NETC', 2:'SNFH', 3:'ET', 4:'RC'})
MODALITIES_IMG   = ['t1n','t1c','t2w','t2f']                      # modalidades de imagen (sin seg)
MOD_LABELS       = {'t1n':'T1n','t1c':'T1c','t2w':'T2w','t2f':'T2-FLAIR'}
MOD_COLORS       = {'t1n':'#3B82F6','t1c':'#EF4444','t2w':'#22C55E','t2f':'#EAB308'}
SUBREGION_COLORS = {'ET':'#3B82F6','NETC':'#EF4444','SNFH':'#22C55E','RC':'#EAB308'}
SANO_COLOR       = '#94A3B8'

# RGBA (0-1) por etiqueta para los overlays de segmentación (no pisa el del EDA original si ya existe)
def _hex_rgba(h, a=0.65):
    h = h.lstrip('#'); return (int(h[0:2],16)/255, int(h[2:4],16)/255, int(h[4:6],16)/255, a)
OVERLAY_RGBA = globals().get('OVERLAY_RGBA', {
    1: _hex_rgba(SUBREGION_COLORS['NETC']),        # NETC
    2: _hex_rgba(SUBREGION_COLORS['SNFH'], 0.55),  # SNFH (edema)
    3: _hex_rgba(SUBREGION_COLORS['ET'],   0.75),  # ET
    4: _hex_rgba(SUBREGION_COLORS['RC']),          # RC
})

# --- Modo de análisis de intensidad (A-E) ---
MODO        = 'completo'   # 'completo' = TODO el dataset (largo) | 'muestra' = SAMPLE_N casos (rápido)
SAMPLE_N    = 25           # casos si MODO == 'muestra'
MAX_CASOS   = None         # tope opcional en modo 'completo' (None = sin tope)

RNG_SEED    = 42
NBINS          = 160       # bins fijos por modalidad para los histogramas acumulados (streaming)
RANGO_SAMPLE_N = 40        # casos de pre-pase para fijar el rango de los bins
RESERVOIR_N    = 12_000    # tope de voxeles por clase para los violines (B)
SUB_RES        = 600       # voxeles aportados por caso al reservoir
MAXVOX_OTSU    = 100_000   # voxeles por caso para Otsu/bimodalidad (C)

# Checkpoints (para no perder avance en corridas largas)
CK_DIR    = DRIVE_DIR if os.path.isdir(DRIVE_DIR) else '.'
CK_STATS  = os.path.join(CK_DIR, 'EDA_intensidad_stats.csv')
CK_OTSU   = os.path.join(CK_DIR, 'EDA_intensidad_otsu.csv')
CK_SEED   = os.path.join(CK_DIR, 'EDA_intensidad_seed.csv')

def _layout_oscuro(fig, titulo, h=400):
    fig.update_layout(title=dict(text=titulo, font=dict(size=14, color='white')),
        paper_bgcolor='#0f172a', plot_bgcolor='#1e293b',
        font=dict(color='white', size=11), height=h,
        legend=dict(bgcolor='#1e293b'))
    fig.update_xaxes(gridcolor='#334155'); fig.update_yaxes(gridcolor='#334155')
    return fig

print('Constantes listas. SAMPLE_N =', SAMPLE_N)

In [ ]:
# === Utilidades ===
def downsample(arr, maxn, seed=RNG_SEED):
    arr = np.asarray(arr).ravel()
    if arr.size <= maxn: return arr
    rng = np.random.default_rng(seed)
    return arr[rng.choice(arr.size, maxn, replace=False)]

def coef_bimodalidad(x):
    x = np.asarray(x); n = x.size
    if n < 4: return np.nan
    g1 = spstats.skew(x); g2 = spstats.kurtosis(x)   # kurtosis en exceso
    denom = g2 + 3.0*((n-1)**2)/((n-2)*(n-3))
    return (g1**2 + 1.0)/denom if denom != 0 else np.nan

def muestra_estratificada(df_vol, n=SAMPLE_N, seed=RNG_SEED):
    d = df_vol[df_vol['TumorTotal'] > 0].copy()
    if len(d) == 0: return df_vol['case_id'].head(n).tolist()
    try:
        d['q'] = pd.qcut(d['TumorTotal'], q=min(4, d['TumorTotal'].nunique()),
                         labels=False, duplicates='drop')
    except Exception:
        d['q'] = 0
    por_q = max(1, n // max(1, d['q'].nunique()))
    samp = d.groupby('q', group_keys=False).apply(
        lambda g: g.sample(min(len(g), por_q), random_state=seed))
    ids = samp['case_id'].tolist()
    for cid in d['case_id'].tolist():
        if len(ids) >= n: break
        if cid not in ids: ids.append(cid)
    return ids[:n]

def _dir_con_casos(base_dir, case_ids):
    if not os.path.isdir(base_dir): return None
    cset = set(case_ids)
    for root, dirs, _ in os.walk(base_dir):
        if cset & set(dirs): return root
    return None

def asegurar_casos(case_ids):
    # Garantiza en disco los NIfTI de case_ids. Usa DATASET_DIR si ya están;
    # si no, extrae SOLO esos casos del ZIP (rápido, no los 2,200).
    base = _dir_con_casos(DATASET_DIR, case_ids)
    faltan = [c for c in case_ids if not (base and os.path.isdir(os.path.join(base, c)))]
    if faltan:
        if not TRAINING_ZIP:
            raise FileNotFoundError(f'Sin datos en {DATASET_DIR} ni ZIP en {DRIVE_DIR}. Ajusta DRIVE_DIR/DATASET_DIR.')
        os.makedirs(DATASET_DIR, exist_ok=True)
        with zipfile.ZipFile(TRAINING_ZIP) as zf:
            miembros = zf.namelist()
            sel = [m for m in miembros if any((f'/{c}/' in m) or m.startswith(f'{c}/') for c in faltan)]
            for m in sel:
                zf.extract(m, DATASET_DIR)
        base = _dir_con_casos(DATASET_DIR, case_ids)
    if base is None:
        raise FileNotFoundError('No se localizaron los casos tras la extracción; revisa el contenido del ZIP.')
    return base

def _ruta(case_id, mod):
    row = df_ok[df_ok['case_id'] == case_id]
    if row.empty: return None
    return row.iloc[0].get(f'{mod}_path')

def cargar_modalidad(case_id, mod):
    return nib.load(_ruta(case_id, mod)).get_fdata().astype(np.float32)

def cargar_seg(case_id):
    return nib.load(_ruta(case_id, 'seg')).get_fdata().astype(np.int16)

# --- Recorrido por casos con extracción bajo demanda (streaming, disco acotado) ---
import shutil

def _centros(edges):
    return (np.asarray(edges)[:-1] + np.asarray(edges)[1:]) / 2

def _construir_miembros_por_caso(zf, case_ids):
    cset = set(case_ids); mp = {}
    for m in zf.namelist():
        for p in m.split('/'):
            if p in cset:
                mp.setdefault(p, []).append(m); break
    return mp

def _paths_en(cp):
    if not cp: return {}
    return {mod: (glob.glob(os.path.join(cp, f'*-{mod}.nii*')) or [None])[0]
            for mod in MODALITIES_IMG + ['seg']}

def recorrer_casos(case_ids):
    """Genera (case_id, dict_paths). Si los casos ya están extraídos en DATASET_DIR los lee;
       si no, extrae uno por uno del ZIP a un temporal y lo borra tras procesarlo."""
    base = _dir_con_casos(DATASET_DIR, case_ids[:50])
    if base is not None and all(os.path.isdir(os.path.join(base, c)) for c in case_ids):
        for cid in case_ids:
            yield cid, _paths_en(os.path.join(base, cid))
        return
    if TRAINING_ZIP is None:
        raise FileNotFoundError('Sin datos extraídos ni ZIP. Ajusta DRIVE_DIR/DATASET_DIR o pre-extrae el dataset.')
    tmp = os.path.join(DATASET_DIR, '_stream_tmp')
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        mp = _construir_miembros_por_caso(zf, case_ids)
        for cid in case_ids:
            miembros = mp.get(cid, [])
            shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp, exist_ok=True)
            for m in miembros:
                try:
                    zf.extract(m, tmp)
                except Exception:
                    pass
            cp = None
            for root, dirs, _ in os.walk(tmp):
                if cid in dirs: cp = os.path.join(root, cid); break
            yield cid, _paths_en(cp)
        shutil.rmtree(tmp, ignore_errors=True)

print('Utilidades listas.')

In [ ]:
# === df_vol + lista de casos (según MODO) + rangos de bins ===
# 1) df_vol: memoria o checkpoint CSV (no toca imágenes).
if 'df_vol' not in globals() or df_vol is None:
    if os.path.exists(VOL_CSV):
        df_vol = pd.read_csv(VOL_CSV)
        print('df_vol cargado del checkpoint:', len(df_vol), 'casos')
    else:
        raise FileNotFoundError(
            'No hay df_vol en memoria ni checkpoint en ' + VOL_CSV + '.\n'
            'Opciones: (a) ejecuta estas celdas anexadas a tu EDA original (que crea df_vol), '
            'o (b) coloca EDA_volumenes_completo.csv en DRIVE_DIR.')
else:
    print('df_vol en memoria:', len(df_vol), 'casos')

# 2) Lista de casos a procesar en A-E.
con_tumor = df_vol[df_vol['TumorTotal'] > 0]['case_id'].tolist()
if MODO == 'muestra':
    CASE_IDS = muestra_estratificada(df_vol, n=SAMPLE_N)
else:
    CASE_IDS = con_tumor if MAX_CASOS is None else con_tumor[:MAX_CASOS]
print(f'MODO = {MODO}  ->  {len(CASE_IDS)} casos para el análisis de intensidad (A-E).')
if MODO == 'completo':
    print('  AVISO: el modo completo lee todas las modalidades de todos los casos; es una corrida larga.')

# 3) Pre-pase ligero para fijar el rango de los bins por modalidad.
rango_ids = muestra_estratificada(df_vol, n=min(RANGO_SAMPLE_N, len(CASE_IDS)))
_acc = {m: [] for m in MODALITIES_IMG}
print(f'Pre-pase de rangos sobre {len(rango_ids)} casos...')
for cid, paths in recorrer_casos(rango_ids):
    for m in MODALITIES_IMG:
        p = paths.get(m)
        if not p: continue
        v = nib.load(p).get_fdata().astype(np.float32); bv = v[v > 0]
        if bv.size: _acc[m].append(downsample(bv, 40_000))
        del v
    gc.collect()
bins_mod = {}
for m in MODALITIES_IMG:
    if _acc[m]:
        a = np.concatenate(_acc[m]); lo, hi = np.percentile(a, [0.5, 99.5])
    else:
        lo, hi = 0.0, 1.0
    bins_mod[m] = np.linspace(float(lo), float(hi), NBINS + 1)
del _acc; gc.collect()
print('Rangos de bins fijados por modalidad.')

In [ ]:
# === Pasada de recolección (streaming sobre CASE_IDS) ===
# Acumula histogramas (conteos en bins fijos) y estadísticas por caso. Memoria acotada.
hist_mod  = {m: np.zeros(NBINS) for m in MODALITIES_IMG}                              # A
hist_sub  = {m: {ln: np.zeros(NBINS) for ln in LABEL_MAP.values()} for m in MODALITIES_IMG}  # B (sep)
hist_sano = {m: np.zeros(NBINS) for m in MODALITIES_IMG}                              # B (sep)
res_sub   = {m: {ln: np.array([]) for ln in LABEL_MAP.values()} for m in MODALITIES_IMG}      # B (violines)
res_sano  = {m: np.array([]) for m in MODALITIES_IMG}                                 # B (violines)
stats_caso, otsu_bimod, semillas = [], [], []                                        # C, D, E
_rng = np.random.default_rng(RNG_SEED)

def _add_res(arr, vals, cap=RESERVOIR_N, sub=SUB_RES):
    s = downsample(vals, sub)
    arr = np.concatenate([arr, s]) if arr.size else s
    if arr.size > cap:
        arr = arr[_rng.choice(arr.size, cap, replace=False)]
    return arr

print(f'Procesando {len(CASE_IDS)} casos en streaming...')
procesados = 0
for cid, paths in recorrer_casos(CASE_IDS):
    segp = paths.get('seg')
    if not segp:
        continue
    try:
        seg = nib.load(segp).get_fdata().astype(np.int16)
    except Exception:
        continue

    mask_et = (seg == 3)
    if mask_et.sum() > 0:
        c = ndimage.center_of_mass(mask_et)
    elif (seg > 0).sum() > 0:
        c = ndimage.center_of_mass(seg > 0)
    else:
        c = None
    ci = tuple(int(round(v)) for v in c) if c is not None else None

    for m in MODALITIES_IMG:
        p = paths.get(m)
        if not p: continue
        try:
            vol = nib.load(p).get_fdata().astype(np.float32)
        except Exception:
            continue
        bm = vol > 0
        bv = vol[bm]
        if bv.size == 0:
            del vol; continue

        hist_mod[m] += np.histogram(bv, bins=bins_mod[m])[0]                          # A
        stats_caso.append({'case_id': cid, 'modalidad': m,                            # D
                           'min': float(bv.min()), 'max': float(bv.max()),
                           'media': float(bv.mean()), 'p99': float(np.percentile(bv, 99))})

        for lv, ln in LABEL_MAP.items():                                              # B
            sv = vol[seg == lv]
            if sv.size:
                hist_sub[m][ln] += np.histogram(sv, bins=bins_mod[m])[0]
                res_sub[m][ln]   = _add_res(res_sub[m][ln], sv)
        sano = vol[bm & (seg == 0)]
        if sano.size:
            hist_sano[m] += np.histogram(sano, bins=bins_mod[m])[0]
            res_sano[m]   = _add_res(res_sano[m], sano)

        bvo = downsample(bv, MAXVOX_OTSU)                                             # C
        rec = {'case_id': cid, 'modalidad': m, 'bimodalidad': coef_bimodalidad(bvo)}
        try:
            rec['otsu_n2_t1'] = float(threshold_multiotsu(bvo, classes=2)[0])
        except Exception:
            rec['otsu_n2_t1'] = np.nan
        try:
            t3 = threshold_multiotsu(bvo, classes=3)
            rec['otsu_n3_t1'], rec['otsu_n3_t2'] = float(t3[0]), float(t3[1])
        except Exception:
            rec['otsu_n3_t1'] = rec['otsu_n3_t2'] = np.nan
        otsu_bimod.append(rec)

        if m == 't1c' and ci is not None:                                            # E
            z, y, x = ci
            z = min(max(z, 0), vol.shape[0]-1); y = min(max(y, 0), vol.shape[1]-1); x = min(max(x, 0), vol.shape[2]-1)
            vecindad = vol[max(0,z-1):z+2, max(0,y-1):y+2, max(0,x-1):x+2]
            semillas.append({'case_id': cid, 'a0': z, 'a1': y, 'a2': x,
                             'intensidad_t1c': float(vol[z, y, x]),
                             'media_vecindad': float(vecindad.mean())})
        del vol, bv
    del seg
    procesados += 1
    if procesados % 25 == 0 or procesados == len(CASE_IDS):
        print(f'  {procesados}/{len(CASE_IDS)} casos')
        try:
            pd.DataFrame(stats_caso).to_csv(CK_STATS, index=False)
            pd.DataFrame(otsu_bimod).to_csv(CK_OTSU, index=False)
            pd.DataFrame(semillas).to_csv(CK_SEED, index=False)
        except Exception:
            pass
        gc.collect()

# Nombres que reutilizan las secciones B (violines) y C/D/E
intens_sub  = res_sub
intens_sano = res_sano
df_stats = pd.DataFrame(stats_caso)
df_otsu  = pd.DataFrame(otsu_bimod)
df_seed  = pd.DataFrame(semillas)
print('Pasada completa:', procesados, 'casos analizados.')

---
## A — Histogramas de intensidad por modalidad

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Distribución de intensidad del cerebro (voxeles &gt; 0, fondo enmascarado) por modalidad. Es la base de Otsu, umbral y K-means: la forma uni/bimodal del histograma anticipa si un umbral global puede separar tejido tumoral del sano.
</div>

In [ ]:
figA = go.Figure()
_presentes = [m for m in MODALITIES_IMG if hist_mod[m].sum() > 0]
for m in _presentes:
    figA.add_trace(go.Bar(x=_centros(bins_mod[m]), y=hist_mod[m], name=MOD_LABELS[m],
        marker_color=MOD_COLORS[m], opacity=0.55))
# Dropdown: 'Todas' u aislar una modalidad
_botones = [dict(label='Todas', method='update', args=[{'visible': [True]*len(_presentes)}])]
for j, m in enumerate(_presentes):
    _botones.append(dict(label=MOD_LABELS[m], method='update',
                         args=[{'visible': [k == j for k in range(len(_presentes))]}]))
figA.update_layout(barmode='overlay', bargap=0,
                   xaxis_title='Intensidad', yaxis_title='Frecuencia (voxeles)',
                   updatemenus=[dict(buttons=_botones, x=1.0, xanchor='right', y=1.18, yanchor='top',
                                     bgcolor='#1e293b', bordercolor='#334155', font=dict(color='white'))])
_layout_oscuro(figA, f'Histograma de intensidad por modalidad — {procesados} casos ({MODO})')
figA.show()

---
## B — Separabilidad de intensidades: sub-región vs tejido sano

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Para cada modalidad se comparan las intensidades de cada sub-región (ET, NETC, SNFH, RC) contra el parénquima sano. La <b>separabilidad</b> se cuantifica con el coeficiente de solapamiento de histogramas (OVL: 0 = no se solapan, 1 = idénticos) y la distancia de Bhattacharyya (mayor = más separable). Esto justifica qué método/umbral conviene por sub-región y modalidad.
</div>

In [ ]:
def ovl_bhatt_hist(ha, hb):
    sa, sb = ha.sum(), hb.sum()
    if sa == 0 or sb == 0: return np.nan, np.nan
    pa, pb = ha/sa, hb/sb
    ovl = float(np.minimum(pa, pb).sum())
    bc  = float(np.sqrt(pa*pb).sum())
    bdist = float(-np.log(bc)) if bc > 0 else np.inf
    return ovl, bdist

# Violines por modalidad (reservoir acotado, representativo)
figB = make_subplots(rows=2, cols=2, subplot_titles=[MOD_LABELS[m] for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i//2 + 1, i % 2 + 1
    sano = intens_sano[m]
    if sano.size:
        figB.add_trace(go.Violin(y=downsample(sano, 8000), name='Sano',
            line_color=SANO_COLOR, showlegend=(i == 0), box_visible=True, meanline_visible=True),
            row=r, col=c)
    for ln in ['ET','NETC','SNFH','RC']:
        v = intens_sub[m][ln]
        if v.size:
            figB.add_trace(go.Violin(y=downsample(v, 8000), name=ln,
                line_color=SUBREGION_COLORS[ln], showlegend=(i == 0),
                box_visible=True, meanline_visible=True), row=r, col=c)
_layout_oscuro(figB, f'Distribución de intensidad por clase y modalidad — {procesados} casos', h=720)
figB.show()

# Tablas de separabilidad (sobre histogramas acumulados de TODOS los casos)
ovl_tab, bh_tab = {}, {}
for m in MODALITIES_IMG:
    ovl_tab[MOD_LABELS[m]] = {}; bh_tab[MOD_LABELS[m]] = {}
    for ln in ['ET','NETC','SNFH','RC']:
        o, b = ovl_bhatt_hist(hist_sub[m][ln], hist_sano[m])
        ovl_tab[MOD_LABELS[m]][ln] = round(o, 3) if o == o else np.nan
        bh_tab[MOD_LABELS[m]][ln]  = round(b, 3) if b == b else np.nan
df_ovl = pd.DataFrame(ovl_tab)
df_bh  = pd.DataFrame(bh_tab)
print('Solapamiento (OVL) sub-región vs sano  —  menor = más separable')
display(df_ovl.style.background_gradient(cmap='RdYlGn_r', axis=None)
        .set_caption('OVL (0=separable, 1=indistinguible)'))
print('\nDistancia de Bhattacharyya  —  mayor = más separable')
display(df_bh.style.background_gradient(cmap='RdYlGn', axis=None)
        .set_caption('Bhattacharyya'))

---
## C — Bimodalidad y vista previa del umbral de Otsu

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Otsu asume un histograma con modas separables. Se calcula el coeficiente de bimodalidad (regla práctica: &gt; 0.555 sugiere bimodalidad) y los umbrales de Otsu para 2 y 3 clases por modalidad. Predice dónde Otsu separará bien (bimodal) y dónde fallará (unimodal).
</div>

In [ ]:
resumen_C = df_otsu.groupby('modalidad').agg(
    bimodalidad=('bimodalidad','median'),
    otsu_n2_t1=('otsu_n2_t1','median'),
    otsu_n3_t1=('otsu_n3_t1','median'),
    otsu_n3_t2=('otsu_n3_t2','median'),
).round(3)
resumen_C['bimodal?'] = resumen_C['bimodalidad'] > 0.555
resumen_C = resumen_C.reindex(MODALITIES_IMG)
resumen_C.index = [MOD_LABELS[m] for m in resumen_C.index]
display(resumen_C.style.set_caption('Bimodalidad y umbrales de Otsu (mediana sobre los casos)'))

# Histogramas acumulados con líneas de umbral Otsu (mediana) por modalidad
figC = make_subplots(rows=2, cols=2, subplot_titles=[MOD_LABELS[m] for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i//2 + 1, i % 2 + 1
    h = hist_mod[m]
    if h.sum() == 0: continue
    figC.add_trace(go.Bar(x=_centros(bins_mod[m]), y=h, marker_color=MOD_COLORS[m],
        opacity=0.7, showlegend=False), row=r, col=c)
    med = df_otsu[df_otsu['modalidad'] == m].median(numeric_only=True)
    for col in ['otsu_n3_t1', 'otsu_n3_t2']:
        th = med.get(col, np.nan)
        if th == th:
            figC.add_vline(x=float(th), line=dict(color='white', width=1.5, dash='dash'), row=r, col=c)
figC.update_layout(bargap=0)
_layout_oscuro(figC, 'Histograma con umbrales de Otsu (n=3, mediana)', h=720)
figC.show()

---
## D — Variabilidad del rango de intensidades entre casos

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Las intensidades de MRI no están en unidades estándar: varían entre casos e instituciones. Si la dispersión de la media/p99 entre casos es alta, un umbral fijo no es transferible y se necesita <b>normalización</b> (z-score o histogram matching) como paso de limpieza antes de segmentar.
</div>

In [ ]:
figD = make_subplots(rows=1, cols=2, subplot_titles=['Media por caso', 'p99 por caso'])
for m in MODALITIES_IMG:
    sub = df_stats[df_stats['modalidad'] == m]
    figD.add_trace(go.Box(y=sub['media'], name=MOD_LABELS[m], marker_color=MOD_COLORS[m],
        showlegend=False, boxmean=True), row=1, col=1)
    figD.add_trace(go.Box(y=sub['p99'], name=MOD_LABELS[m], marker_color=MOD_COLORS[m],
        showlegend=False, boxmean=True), row=1, col=2)
_layout_oscuro(figD, 'Dispersión de intensidad entre casos (motiva normalización)')
figD.show()

tab_var = df_stats.groupby('modalidad')[['media','p99']].agg(['mean','std']).round(2)
tab_var['CV_media_%'] = (df_stats.groupby('modalidad')['media'].std() /
                         df_stats.groupby('modalidad')['media'].mean() * 100).round(1)
tab_var.index = [MOD_LABELS[m] for m in tab_var.index]
display(tab_var)

---
## E — Análisis de semillas para crecimiento de regiones

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Para <code>ConnectedThreshold</code> / <code>ConfidenceConnected</code> hay que elegir una semilla y un rango de tolerancia. Se mide la intensidad en T1c en el centroide del tumor (ET) entre casos. El rango p10–p90 sugiere ventanas razonables de intensidad de semilla, y la dispersión orienta la tolerancia.
</div>

In [ ]:
if len(df_seed) > 0:
    s = df_seed['intensidad_t1c']
    p10, p50, p90 = np.percentile(s, [10, 50, 90])
    figE = go.Figure()
    figE.add_trace(go.Histogram(x=s, nbinsx=25, marker_color='#EF4444', opacity=0.8,
        name='Intensidad en centroide (T1c)'))
    for x, lbl in [(p10,'p10'), (p50,'mediana'), (p90,'p90')]:
        figE.add_vline(x=float(x), line=dict(color='white', dash='dash'),
                       annotation_text=lbl, annotation_font_color='white')
    figE.update_layout(xaxis_title='Intensidad T1c en el centroide del tumor', yaxis_title='Casos')
    _layout_oscuro(figE, 'Intensidad de semilla candidata (T1c) entre casos')
    figE.show()
    tol = float(df_seed['intensidad_t1c'].std())
    print(f'Rango de intensidad de semilla (T1c): p10={p10:.1f}  mediana={p50:.1f}  p90={p90:.1f}')
    print(f'Tolerancia sugerida para ConnectedThreshold (~1 sigma): +/- {tol:.1f}')
    display(df_seed.round(1).head(15))
else:
    print('Sin datos de semilla (no se encontró ET ni tumor en la muestra).')

---
## F — Subconjunto representativo de casos para el prototipo

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
El prototipo clásico no corre sobre los ~2,200 casos: se eligen 4–6 casos demostrativos que cubran los retos. Se exporta <code>casos_demostrativos.csv</code> para que <b>todo el equipo segmente y visualice los mismos casos</b>.
</div>

In [ ]:
dv = df_vol[df_vol['TumorTotal'] > 0].copy()
sel = {}
def _pick(mask_series, etiqueta, criterio):
    cand = dv[mask_series]
    if len(cand) == 0: return
    cid = criterio(cand)
    sel.setdefault(cid, []).append(etiqueta)

# típico (mediana de TumorTotal)
med = dv['TumorTotal'].median()
_pick(pd.Series(True, index=dv.index), 'Típico (mediana de tumor)',
      lambda c: c.iloc[(c['TumorTotal']-med).abs().argsort()].iloc[0]['case_id'])
# grande (p95) y pequeño (p5)
_pick(pd.Series(True, index=dv.index), 'Tumor grande (~p95)',
      lambda c: c.sort_values('TumorTotal').iloc[int(len(c)*0.95)-1]['case_id'])
_pick(pd.Series(True, index=dv.index), 'Tumor pequeño (~p5)',
      lambda c: c.sort_values('TumorTotal').iloc[int(len(c)*0.05)]['case_id'])
# post-resección (RC > 0)
_pick(dv['RC'] > 0, 'Post-resección (RC>0)',
      lambda c: c.sort_values('RC', ascending=False).iloc[0]['case_id'])
# solo edema (ET == 0, SNFH > 0)
_pick((dv['ET'] == 0) & (dv['SNFH'] > 0), 'Solo edema (sin ET)',
      lambda c: c.sort_values('SNFH', ascending=False).iloc[0]['case_id'])
# NETC dominante
_pick(dv['NETC'] > 0, 'NETC dominante',
      lambda c: c.assign(r=c['NETC']/c['TumorTotal']).sort_values('r', ascending=False).iloc[0]['case_id'])

rows = []
for cid, etiquetas in sel.items():
    r = df_vol[df_vol['case_id'] == cid].iloc[0]
    rows.append({'case_id': cid, 'motivo': ' / '.join(etiquetas),
                 'TumorTotal_cc': r['TumorTotal'],
                 **{ln: r[ln] for ln in LABEL_MAP.values()}})
df_demo = pd.DataFrame(rows)
out_csv = 'casos_demostrativos.csv'
df_demo.to_csv(out_csv, index=False)
print('Casos demostrativos guardados en', os.path.abspath(out_csv))
display(df_demo)

---
## Visor de cortes (casos demostrativos)

Renderiza cortes axiales centrados en el tumor de los casos de la sección F, como imágenes embebidas (base64) para el visor interactivo del HTML (slider de corte, botones de modalidad y overlay on/off).

In [ ]:
from PIL import Image
import io, base64

N_SLICES_VISOR = 9     # cortes axiales centrados en el tumor
MINIATURA      = 220   # px

def _prep(arr):
    return np.rot90(np.asarray(arr))   # axial en vertical

def _norm8(sl):
    sl = np.asarray(sl, np.float32); m = sl > 0
    lo, hi = (np.percentile(sl[m], [1, 99]) if m.sum() else (0.0, 1.0))
    if hi <= lo: hi = lo + 1
    out = np.clip((sl - lo)/(hi - lo), 0, 1); out[~m] = 0
    return (out*255).astype(np.uint8)

def _b64(img, fmt, q=72):
    buf = io.BytesIO()
    img.save(buf, format=fmt, quality=q) if fmt == 'JPEG' else img.save(buf, format=fmt)
    return f'data:image/{fmt.lower()};base64,' + base64.b64encode(buf.getvalue()).decode()

def _overlay_rgba(seg_sl):
    s = _prep(seg_sl); h, w = s.shape
    rgba = np.zeros((h, w, 4), np.uint8)
    for lv, (r, g, b, a) in OVERLAY_RGBA.items():
        rgba[s == lv] = [int(r*255), int(g*255), int(b*255), int(a*255)]
    return rgba

viewer_data = {}
demo_ids = df_demo['case_id'].tolist()
print('Renderizando visor para', len(demo_ids), 'casos...')
for cid, paths in recorrer_casos(demo_ids):
    if not paths.get('seg'): continue
    seg = nib.load(paths['seg']).get_fdata().astype(np.int16)
    if (seg > 0).sum() == 0:
        del seg; continue
    c = ndimage.center_of_mass(seg == 3) if (seg == 3).sum() > 0 else ndimage.center_of_mass(seg > 0)
    zc = int(round(c[2])); nz = seg.shape[2]; half = N_SLICES_VISOR // 2
    zlist = [z for z in range(zc - half, zc + half + 1) if 0 <= z < nz]
    entry = {'z': zlist, 'overlay': [], 'mods': {}}
    for z in zlist:
        img = Image.fromarray(_overlay_rgba(seg[:, :, z]), 'RGBA').resize((MINIATURA, MINIATURA), Image.NEAREST)
        entry['overlay'].append(_b64(img, 'PNG'))
    for m in MODALITIES_IMG:
        p = paths.get(m)
        if not p:
            entry['mods'][MOD_LABELS[m]] = []; continue
        vol = nib.load(p).get_fdata().astype(np.float32)
        lst = []
        for z in zlist:
            img = Image.fromarray(_norm8(_prep(vol[:, :, z])), 'L').resize((MINIATURA, MINIATURA), Image.BILINEAR)
            lst.append(_b64(img, 'JPEG'))
        entry['mods'][MOD_LABELS[m]] = lst
        del vol
    viewer_data[cid] = entry
    del seg; gc.collect()
print('Visor listo:', len(viewer_data), 'casos con cortes.')

---
## Nota sobre S6 (parches) y S10 (pesos de pérdida) de la entrega 2

Esas dos secciones del EDA original están orientadas a **aprendizaje profundo** (muestreo por parches y pesos de clase para la función de pérdida), enfoque que **no aplica a este proyecto** (segmentación clásica con ITK). Su contenido se reinterpreta así:

- **Desbalance de clases (S6/S10) → prioridad de sub-región para métodos clásicos.** Las sub-regiones minoritarias (ET, NETC) son las más difíciles de aislar por intensidad; las secciones B y C indican con qué modalidad y umbral abordarlas.
- **No se reportan pesos de pérdida ni diseño de red.** El "diseño" aquí es la elección del método clásico por sub-región (umbral/Otsu para ET en T1c, crecimiento de regiones con la semilla de la sección E, watershed/K-means para regiones difusas).

---
## G — Salidas para la etapa de registro

Verifica la geometría de los volúmenes leyendo **solo los headers NIfTI** (rápido, sin cargar voxeles). Responde lo que registro necesita saber antes de empezar: ¿las modalidades ya están coregistradas entre sí?, ¿el espaciado es isotrópico 1mm?, ¿qué modalidad fija y spacing usar? Exporta `geometria_casos.csv` y `parametros_registro.json`.

In [ ]:
import zlib, io, json

MAX_GEO = None   # None = todos los casos (header-only es barato); pon un tope para acelerar

def _geom(shape, zooms, aff):
    shape = tuple(int(s) for s in shape[:3])
    zooms = tuple(round(float(z), 3) for z in zooms[:3])
    aff   = np.round(np.asarray(aff, float), 3)
    return shape, zooms, ''.join(nib.aff2axcodes(aff)), aff

def _header_zip(zf, member):
    with zf.open(member) as fp:
        comp = fp.read(65536)               # solo el inicio del .nii.gz
    crudo = zlib.decompressobj(31).decompress(comp, 400)
    return nib.Nifti1Header.from_fileobj(io.BytesIO(crudo))

geo_ids = df_vol['case_id'].tolist()
if MAX_GEO is not None:
    geo_ids = geo_ids[:MAX_GEO]

base = _dir_con_casos(DATASET_DIR, geo_ids[:50])
zf   = zipfile.ZipFile(TRAINING_ZIP) if (TRAINING_ZIP and base is None) else None
mp   = _construir_miembros_por_caso(zf, geo_ids) if zf else {}

geo_rows = []
print(f'Leyendo geometría (headers) de {len(geo_ids)} casos...')
for k, cid in enumerate(geo_ids, 1):
    metas = {}
    cp = os.path.join(base, cid) if base else None
    for mod in MODALITIES_IMG + ['seg']:
        try:
            if cp and os.path.isdir(cp):
                hits = glob.glob(os.path.join(cp, f'*-{mod}.nii*'))
                if hits:
                    im = nib.load(hits[0])
                    metas[mod] = _geom(im.shape, im.header.get_zooms(), im.affine)
            elif zf is not None:
                mem = next((m for m in mp.get(cid, []) if f'-{mod}.nii' in m), None)
                if mem:
                    h = _header_zip(zf, mem)
                    metas[mod] = _geom(h.get_data_shape(), h.get_zooms(), h.get_best_affine())
        except Exception:
            pass
    if not metas:
        continue
    shape0, zooms0, orient0, aff0 = next(iter(metas.values()))
    intra = all(v[0] == shape0 and np.allclose(v[3], aff0, atol=1e-2) for v in metas.values())
    iso   = (max(zooms0) - min(zooms0) < 0.05) and (abs(zooms0[0] - 1.0) < 0.05)
    geo_rows.append({'case_id': cid,
                     'shape': 'x'.join(map(str, shape0)),
                     'spacing': 'x'.join(f'{z:.2f}' for z in zooms0),
                     'orientacion': orient0,
                     'intra_coregistrado': bool(intra),
                     'iso_1mm': bool(iso)})
    if k % 250 == 0:
        print(f'  {k}/{len(geo_ids)}')
if zf is not None:
    zf.close()

df_geo = pd.DataFrame(geo_rows)
pct_intra = 100*df_geo['intra_coregistrado'].mean() if len(df_geo) else 0.0
pct_iso   = 100*df_geo['iso_1mm'].mean() if len(df_geo) else 0.0
necesita_intra = pct_intra <= 95
recomienda_atlas = pct_intra > 95

# Modalidad fija = la de mejor separabilidad de ET vs sano (menor OVL en la fila 'ET')
try:
    mod_fija = df_ovl.loc['ET'].astype(float).idxmin()
except Exception:
    mod_fija = 'T1c'
spacing_obj = df_geo['spacing'].mode().iat[0] if len(df_geo) else '1.00x1.00x1.00'

if necesita_intra:
    CONCLUSION = ('Hay casos sin coregistrar entre modalidades: se requiere coregistro multimodal '
                  'intra-sujeto (información mutua + rígida) antes de segmentar.')
else:
    CONCLUSION = ('Las modalidades ya vienen coregistradas entre sí; el coregistro intra-sujeto es '
                  'redundante. El registro aporta valor como registro a un atlas/plantilla o como '
                  'demostración de la técnica sobre los casos demostrativos.')

params = {
    'modalidad_fija_recomendada': str(mod_fija),
    'spacing_objetivo': spacing_obj,
    'metrica_recomendada': 'Informacion mutua (Mattes) - multimodal',
    'transformacion': 'Rigida (Euler3D / VersorRigid3D)',
    'registro_intra_sujeto_necesario': bool(necesita_intra),
    'registro_a_atlas_recomendado': bool(recomienda_atlas),
    'pct_casos_coregistrados': round(pct_intra, 1),
    'pct_casos_iso_1mm': round(pct_iso, 1),
    'casos_demostrativos': df_demo['case_id'].tolist(),
    'conclusion': CONCLUSION,
}

df_registro_resumen = pd.DataFrame([
    ('Modalidad fija recomendada', str(mod_fija)),
    ('Spacing objetivo', spacing_obj),
    ('Métrica recomendada', 'Información mutua (Mattes) — multimodal'),
    ('Transformación', 'Rígida (Euler3D / VersorRigid3D)'),
    ('% casos con modalidades coregistradas', f'{pct_intra:.1f}%'),
    ('% casos isotrópicos 1mm', f'{pct_iso:.1f}%'),
    ('¿Registro intra-sujeto necesario?', 'Sí' if necesita_intra else 'No'),
    ('¿Registro a atlas recomendado?', 'Sí' if recomienda_atlas else 'No'),
    ('Conclusión', CONCLUSION),
], columns=['Aspecto', 'Valor'])

# Exportar (Drive si está disponible, y copia local)
for d in {CK_DIR, '.'}:
    try:
        df_geo.to_csv(os.path.join(d, 'geometria_casos.csv'), index=False)
        with open(os.path.join(d, 'parametros_registro.json'), 'w', encoding='utf-8') as fjson:
            json.dump(params, fjson, ensure_ascii=False, indent=2)
    except Exception:
        pass

print('geometria_casos.csv y parametros_registro.json generados.')
print(f'Coregistrados intra-sujeto: {pct_intra:.1f}%  |  isotrópico 1mm: {pct_iso:.1f}%')
print('Conclusión:', CONCLUSION)
display(df_registro_resumen)
display(df_geo.head(12))

---
## Reporte HTML autocontenido (A–F)

Genera un HTML con todas las figuras y tablas de este complemento, en el mismo estilo oscuro. Es el entregable clave de la tarea 1.

In [ ]:
import plotly.io as pio, json

def fig_html(fig, incluir_js):
    return pio.to_html(fig, full_html=False,
                       include_plotlyjs=('inline' if incluir_js else False),
                       default_width='100%', default_height='460px')

def tabla_html(df, titulo):
    return (f"<div class='stitle'>{titulo}</div>" +
            df.to_html(border=0, classes='tbl', float_format=lambda x: f'{x:.3f}'))

def tabla_interactiva(df, titulo):
    ths = ''.join(f"<th onclick='_ordenar(this)'>{c} &#8597;</th>" for c in df.columns)
    trs = ''
    for _, row in df.iterrows():
        tds = ''.join(f"<td>{(format(v, '.3f') if isinstance(v, float) else v)}</td>" for v in row)
        trs += f"<tr>{tds}</tr>"
    return (f"<div class='stitle'>{titulo}</div>"
            f"<input class='busca' onkeyup='_filtrar(this)' placeholder='Filtrar...'>"
            f"<table class='tbl tint'><thead><tr>{ths}</tr></thead><tbody>{trs}</tbody></table>")

# (titulo, figuras, [(subtitulo, df, interactiva)])
secciones = [
    ('A — Histograma de intensidad por modalidad', [figA], []),
    ('B — Separabilidad sub-región vs sano', [figB],
        [('Solapamiento (OVL) — menor = más separable', df_ovl, False),
         ('Distancia de Bhattacharyya — mayor = más separable', df_bh, False)]),
    ('C — Bimodalidad y umbrales de Otsu', [figC],
        [('Bimodalidad y Otsu (mediana)', resumen_C, False)]),
    ('D — Variabilidad de intensidad entre casos', [figD], [('Resumen de variabilidad', tab_var, False)]),
]
if len(df_seed) > 0:
    secciones.append(('E — Semillas para crecimiento de regiones', [figE], []))
secciones.append(('F — Subconjunto representativo', [], [('Casos demostrativos (ordena/filtra)', df_demo, True)]))
_g_tablas = [('Resumen y parámetros de registro', df_registro_resumen, False)]
if len(df_geo) > 0:
    _g_tablas.append(('Geometría por caso (ordena/filtra)', df_geo, True))
secciones.append(('G — Salidas para registro', [], _g_tablas))

partes, primero = [], True
for idx, (titulo, figs, tablas) in enumerate(secciones):
    partes.append(f"<section id='s{idx}'><h2>{titulo}</h2>")
    for f in figs:
        partes.append(fig_html(f, primero)); primero = False
    for tt, df, inter in tablas:
        partes.append(tabla_interactiva(df, tt) if inter else tabla_html(df, tt))
    partes.append("</section>")
secciones_html = "".join(partes)

# --- Sección del visor de cortes ---
vidx = len(secciones)
if viewer_data:
    _ops = ''.join(f"<option value='{cid}'>{cid}</option>" for cid in viewer_data)
    _mbtns = ''.join(f"<button class='mbtn' onclick='visorMod(this)'>{MOD_LABELS[m]}</button>" for m in MODALITIES_IMG)
    visor_section = (
        f"<section id='s{vidx}'><h2>Visor de cortes — casos demostrativos</h2>"
        "<div class='vctrl'>"
        f"<label>Caso <select id='vcaso' onchange='visorCaso()'>{_ops}</select></label> &nbsp; "
        f"<span id='vmods'>{_mbtns}</span> &nbsp; "
        "<label><input type='checkbox' id='voverlay' checked onchange='visorRender()'> Overlay</label>"
        "</div>"
        "<div class='vstage'><img id='vimg'><img id='vov'></div>"
        "<input type='range' id='vslice' min='0' max='0' value='0' oninput='visorRender()'>"
        "<div id='vinfo' class='stitle'></div></section>")
else:
    visor_section = ""

nav_items = [(i, s[0].split(' — ')[0]) for i, s in enumerate(secciones)]
if viewer_data: nav_items.append((vidx, 'Visor'))
nav = "".join(f"<a href='#s{i}'>{lbl}</a>" for i, lbl in nav_items)

ESTILO = '''<style>
 body{background:#0f172a;color:#e2e8f0;font-family:system-ui,Arial;margin:0;padding:0 0 60px}
 header{background:#0f172a;padding:28px 32px;border-bottom:1px solid #1e293b}
 header h1{margin:0;font-size:22px} header p{color:#94a3b8;margin:6px 0 0}
 nav{position:sticky;top:0;background:#0f172a;padding:10px 32px;border-bottom:1px solid #1e293b;z-index:5}
 nav a{color:#60a5fa;margin-right:14px;text-decoration:none;font-size:13px}
 section{padding:22px 32px;border-bottom:1px solid #1e293b}
 section h2{font-size:17px;color:#fff}
 .stitle{color:#94a3b8;font-size:13px;margin:14px 0 6px}
 table.tbl{border-collapse:collapse;font-size:12px;margin:4px 0 14px}
 table.tbl th{background:#1e293b;color:#fff;padding:7px 12px;text-align:left}
 table.tbl td{padding:6px 12px;border-top:1px solid #1e293b;color:#cbd5e1}
 table.tint th{cursor:pointer;user-select:none}
 .busca{background:#1e293b;color:#e2e8f0;border:1px solid #334155;border-radius:6px;padding:5px 10px;margin:4px 0;width:240px;display:block}
 select{background:#1e293b;color:#e2e8f0;border:1px solid #334155;border-radius:6px;padding:4px}
 .vctrl{margin:8px 0}
 .mbtn{background:#1e293b;color:#cbd5e1;border:1px solid #334155;border-radius:6px;padding:4px 10px;margin:0 2px;cursor:pointer}
 .mbtn.act{background:#2563eb;color:#fff;border-color:#2563eb}
 .vstage{position:relative;width:260px;height:260px;background:#000;margin:10px 0;border:1px solid #1e293b}
 .vstage img{position:absolute;top:0;left:0;width:260px;height:260px}
 #vov{pointer-events:none}
 #vslice{width:260px}
</style>'''

SCRIPT = '''<script>
function _filtrar(inp){
  var t = inp.nextElementSibling, f = inp.value.toLowerCase();
  t.querySelectorAll('tbody tr').forEach(function(r){
    r.style.display = r.innerText.toLowerCase().indexOf(f) > -1 ? '' : 'none';
  });
}
function _ordenar(th){
  var table = th.closest('table'), tb = table.querySelector('tbody');
  var col = Array.prototype.indexOf.call(th.parentNode.children, th);
  var asc = th.getAttribute('data-asc') !== 'true';
  th.setAttribute('data-asc', asc);
  var rows = Array.prototype.slice.call(tb.querySelectorAll('tr'));
  rows.sort(function(a,b){
    var x = a.children[col].innerText, y = b.children[col].innerText;
    var nx = parseFloat(x), ny = parseFloat(y);
    if(!isNaN(nx) && !isNaN(ny)){ return (nx-ny)*(asc?1:-1); }
    return (x>y?1:(x<y?-1:0))*(asc?1:-1);
  });
  rows.forEach(function(r){ tb.appendChild(r); });
}
var V_MOD = 'T1c';
function visorCaso(){
  var cid = document.getElementById('vcaso').value, z = VIEWER[cid].z;
  var sl = document.getElementById('vslice');
  sl.max = z.length - 1; sl.value = Math.floor(z.length/2);
  visorRender();
}
function visorMod(btn){
  V_MOD = btn.innerText;
  document.querySelectorAll('#vmods .mbtn').forEach(function(b){ b.classList.toggle('act', b===btn); });
  visorRender();
}
function visorRender(){
  var cid = document.getElementById('vcaso').value;
  var i = parseInt(document.getElementById('vslice').value), d = VIEWER[cid];
  var mods = d.mods[V_MOD] || [];
  document.getElementById('vimg').src = mods[i] || '';
  var ov = document.getElementById('vov');
  ov.src = d.overlay[i] || '';
  ov.style.opacity = document.getElementById('voverlay').checked ? '1' : '0';
  document.getElementById('vinfo').innerText = cid + '  ·  ' + V_MOD + '  ·  corte axial z=' + d.z[i];
}
window.addEventListener('load', function(){
  document.querySelectorAll('#vmods .mbtn').forEach(function(b){ if(b.innerText==='T1c') b.classList.add('act'); });
  if(document.getElementById('vcaso')) visorCaso();
});
</script>'''

header = ("<header><h1>EDA — Análisis de intensidad para segmentación clásica</h1>"
          f"<p>BraTS 2024 GLI · {procesados} casos analizados ({MODO}) · "
          "Abel Albuez, Victoria Acero, Santiago Gil</p></header>")

html = ("<!doctype html><html lang='es'><head><meta charset='utf-8'>"
        "<title>EDA intensidad — BraTS 2024 GLI</title>" + ESTILO + "</head><body>"
        + header + "<nav>" + nav + "</nav>"
        + secciones_html + visor_section
        + "<script>var VIEWER=" + json.dumps(viewer_data) + ";</script>"
        + SCRIPT + "</body></html>")

out_html = 'EDA_BraTS2024_GLI_reporte_clasico.html'
with open(out_html, 'w', encoding='utf-8') as f:
    f.write(html)
tam = os.path.getsize(out_html)/1e6
print(f'Reporte HTML interactivo guardado en {os.path.abspath(out_html)} ({tam:.1f} MB)')
print('La descarga de todas las salidas se hace en la celda final (Descargas).')

---
## Descargas

Junta todas las salidas del EDA (reporte HTML + CSV + JSON) en un ZIP y lo descarga.

In [ ]:
import zipfile
try:
    from google.colab import files; _EN_COLAB = True
except Exception:
    _EN_COLAB = False

_dirs = []
for d in ['.', globals().get('CK_DIR', '.'), globals().get('DRIVE_DIR', '')]:
    if d and d not in _dirs: _dirs.append(d)

_nombres = ['EDA_BraTS2024_GLI_reporte_clasico.html',
            'casos_demostrativos.csv', 'geometria_casos.csv', 'parametros_registro.json',
            'EDA_intensidad_stats.csv', 'EDA_intensidad_otsu.csv', 'EDA_intensidad_seed.csv']

encontrados = []
for n in _nombres:
    for d in _dirs:
        p = os.path.join(d, n)
        if os.path.exists(p):
            encontrados.append(p); break

print('Salidas encontradas:')
for p in encontrados: print('  -', p)

zip_out = 'EDA_BraTS2024_GLI_salidas.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in encontrados:
        zf.write(p, os.path.basename(p))
print('ZIP listo:', os.path.abspath(zip_out), round(os.path.getsize(zip_out)/1e6, 2), 'MB')

# Descarga automática (ZIP con todo) + el HTML suelto por comodidad
if _EN_COLAB:
    files.download(zip_out)
    if os.path.exists('EDA_BraTS2024_GLI_reporte_clasico.html'):
        files.download('EDA_BraTS2024_GLI_reporte_clasico.html')